# HW9: Supervised Learning with Spark MLlib

**ST554**
**David Pressley**

## Predicting Heart Disease with the UCI Cleveland Dataset

Use PySpark MLlib to fit and compare three classification models for predicting heart disease. The data comes from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/45/heart+disease) and includes 303 patient records from the Cleveland Clinic with 13 clinical features.

Predict whether a patient has heart disease (binary: present vs. absent) using three different model classes:

1. Logistic Regression
2. Decision Tree
3. Random Forest

Each model will be tuned with cross-validation, evaluated using AUC (Area Under the ROC Curve), and compared to select an overall best.

## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("hw9").getOrCreate()

The UCI Cleveland dataset doesn't include column headers, so we assign them manually. Missing values are encoded as `?` in the raw file. The original target ranges from 0 (no disease) to 4, so we collapse it into a binary variable: 0 stays 0, and anything greater than 0 becomes 1.

In [ ]:
col_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
pdf = pd.read_csv(url, names=col_names, na_values='?')

# a handful of rows have missing values in ca and thal
print(f"Shape before dropping NAs: {pdf.shape}")
pdf = pdf.dropna()
print(f"Shape after dropping NAs: {pdf.shape}")

# binarize target and rename to label
pdf['target'] = (pdf['target'] > 0).astype(int)
pdf = pdf.rename(columns={'target': 'label'})

# convert to Spark DataFrame
df = spark.createDataFrame(pdf)
df.show(5)

## Exploratory Data Analysis

In [ ]:
df.printSchema()

In [ ]:
df.describe().show()

In [ ]:
df.groupBy('label').count().show()

The classes are reasonably balanced. Here's a quick rundown of the 13 predictor features:

- **age**: age in years
- **sex**: 1 = male, 0 = female
- **cp**: chest pain type (0-3)
- **trestbps**: resting blood pressure (mm Hg)
- **chol**: serum cholesterol (mg/dl)
- **fbs**: fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
- **restecg**: resting ECG results (0-2)
- **thalach**: maximum heart rate achieved
- **exang**: exercise-induced angina (1 = yes, 0 = no)
- **oldpeak**: ST depression induced by exercise
- **slope**: slope of the peak exercise ST segment (0-2)
- **ca**: number of major vessels colored by fluoroscopy (0-3)
- **thal**: thalassemia (3 = normal, 6 = fixed defect, 7 = reversible defect)

## Splitting the Data

We split 70/30 into training and test sets using MLlib's `randomSplit()`. The training set is used for cross-validation and hyperparameter tuning. The test set is held out entirely until the end for final model comparison.

In [ ]:
train, test = df.randomSplit([0.7, 0.3], seed=42)
print(f"Training set size: {train.count()}")
print(f"Test set size: {test.count()}")

## Evaluation Metric: AUC

Use AUC (Area Under the ROC Curve) to judge all three models. AUC measures how well a classifier can separate the two classes for each classification threshold by plotting true positive rate against false positive rate for each classification threshold.

AUC ranges from 0 to 1. A value of 1.0 means perfect separation between classes, while 0.5 means the model is no better than a coin flip.

AUC is a good fit because it evaluates the model's overall ranking ability rather than depending on a specific cutoff. Additionally, in a medical context, we care about how well the model ranks patients by risk, and AUC captures exactly that.

## Model Descriptions

Below are conceptual descriptions of each model class we'll be fitting. No code here, just the ideas behind what each model is doing.

### Model 1: Logistic Regression

Logistic regression is a linear model for binary classification. It takes a linear combination of the features and passes the result through the logistic function, which maps any real number to a value between 0 and 1. That output is interpreted as the probability that an observation belongs to the positive class. In our case, that's the probability that heart disease is present.

The model learns its coefficients by maximizing the likelihood of the observed data. To prevent overfitting, we can add regularization, which penalizes large coefficient values. L1 regularization (Lasso) tends to push some coefficients all the way to zero, which basically performs variable selection. L2 regularization (Ridge) shrinks coefficients toward zero without eliminating them. Elastic Net is a blend of the two.

Because logistic regression assumes a linearity, it works best when the relationship between the features and the outcome is roughly linear.You can look at the coefficients directly to understand how each feature contributes to the prediction. It's also fast to train and scales well.

### Model 2: Decision Tree

A decision tree is a model that classifies observations by splitting the data into progressively smaller groups through a series of binary "questions"f about the features. At each step, the algorithm picks a feature and a threshold (for example, "Is age > 55?") and divides the data into two groups based on the answer. The split is chosen to make the resulting groups as homogeneous as possible with respect to the response variable.

The way it measures homogeneity depends on the impurity criterion. For classification, two common choices are Gini impurity and entropy. Both measure how mixed the classes are in a group. A group with all one class has zero impurity, and a group that's evenly split between classes has maximum impurity. At each step, the tree picks the split that reduces impurity the most.

Splitting continues recursively until some stopping condition is reached, such as a maximum tree depth or a minimum number of observations in a leaf node. The final groups each get a classification label based on the majority of the training observations that ended up in that leaf.

Decision trees are easy to interpret. You can trace the path from the root to a leaf and see exactly which feature values led to a given prediction. They also handle non-linear relationships naturally. The main downside is that they tend to overfit, especially when they grow deep. Small changes in the training data can produce very different trees.

### Model 3: Random Forest

A random forest is an ensemble method that addresses overfitting of individual decision trees by combining many of them. The core idea is that while any single tree might be overfit, averaging across a large number of diverse trees tends to cancel out the noise and produce a more stable, accurate prediction.

Each tree is trained on a bootstraped sample. A random draw is taken from the training data, so each tree sees a slightly different version of the data. This is called bagging (bootstrap aggregating). Then, at each split, the tree only considers a random subset of the available features rather than all of them. This forces the trees to differ from one another even further.

A random forest is essentially a collection of decision trees that have been deliberately made different from one another, and then averaged to reduce variance.

Random forests handle non-linear relationships naturally, are robust to outliers and irrelevant features, and generally perform well out of the box. The tradeoff compared to a single decision tree is that you lose the ability to trace a single clear decision path, since the prediction is an aggregate of many trees.